In [37]:
# ! pip install -U langchain langchain-core langchain-openai langgraph

https://openweathermap.org/api/current?collection=current_forecast

In [ ]:
"""
날씨 확인 챗봇 - LangGraph ReAct Agent + OpenRouter
OpenWeatherMap API를 통해 실시간 날씨 정보를 제공합니다.

설치:
    pip install langgraph langchain-core langchain-openai requests python-dotenv

환경변수 (.env):
    OPENROUTER_API_KEY=...
    OPENWEATHER_API_KEY=...   # https://openweathermap.org/api (무료)
"""

import os
import requests
from collections import defaultdict
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# ──────────────────────────────────────────────
# 환경변수
# ──────────────────────────────────────────────
load_dotenv(override=True)

OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL      = "https://openrouter.ai/api/v1"

ELICE_API_KEY       = os.getenv("ELICE_API_KEY")
ELICE_URL           = os.getenv("ELICE_URL")

OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

# MODEL_NAME          = "openai/gpt-oss-120b:free"
# MODEL_NAME          = "nvidia/nemotron-3-super-120b-a12b:free"
MODEL_NAME          = "zai-org/glm-4.7"

# ──────────────────────────────────────────────
# OpenWeatherMap 헬퍼
# ──────────────────────────────────────────────
OWM_BASE = "https://api.openweathermap.org/data/2.5"
OWM_GEO  = "https://api.openweathermap.org/geo/1.0"


CONDITION_KO = {
    "Clear"       : "맑음 ☀️",
    "Clouds"      : "흐림 ☁️",
    "Rain"        : "비 🌧️",
    "Drizzle"     : "이슬비 🌦️",
    "Thunderstorm": "천둥번개 ⛈️",
    "Snow"        : "눈 ❄️",
    "Mist"        : "안개 🌫️",
    "Fog"         : "짙은 안개 🌁",
    "Haze"        : "연무 🌫️",
    "Dust"        : "황사 🌪️",
    "Sand"        : "모래바람 🌪️",
    "Smoke"       : "연기 💨",
    "Tornado"     : "토네이도 🌪️",
}


def _geocode(city: str):
    """도시명 → (위도, 경도, 표시명). 실패 시 None."""
    if not OPENWEATHER_API_KEY:
        return None
    try:
        r = requests.get(
            f"{OWM_GEO}/direct",
            params={"q": city, "limit": 5, "appid": OPENWEATHER_API_KEY},
            # timeout=10,
        )
        r.raise_for_status()
        data = r.json()
        if not data:
            return None
        loc  = data[0]
        name = loc.get("local_names", {}).get("ko") or loc["name"]
        print(name, loc["lat"], loc["lon"])
        return loc["lat"], loc["lon"], name
    except Exception:
        return None


def _fmt_current(data: dict, city: str) -> str:
    m    = data["main"]
    w    = data["weather"][0]
    wnd  = data.get("wind", {})
    cond = CONDITION_KO.get(w["main"], w["description"])
    vis  = data.get("visibility")
    lines = [
        f"📍 {city} 현재 날씨",
        f"  날씨     : {cond}",
        f"  기온     : {m['temp']:.1f}°C  (체감 {m['feels_like']:.1f}°C)",
        f"  최저/최고 : {m['temp_min']:.1f}°C / {m['temp_max']:.1f}°C",
        f"  습도     : {m['humidity']}%",
        f"  풍속     : {wnd.get('speed', 0):.1f} m/s",
    ]
    if vis is not None:
        lines.append(f"  가시거리  : {vis / 1000:.1f} km")
    return "\n".join(lines)


def _fmt_forecast(data: dict, city: str, days: int) -> str:
    daily: dict[str, list] = defaultdict(list)
    for item in data["list"]:
        date = item["dt_txt"].split()[0]
        daily[date].append(item)

    lines = [f"📅 {city} {days}일 예보"]
    for date, items in list(daily.items())[:days]:
        temps = [i["main"]["temp"] for i in items]
        conds = [i["weather"][0]["main"] for i in items]
        top   = max(set(conds), key=conds.count)
        lines.append(
            f"  {date}: {CONDITION_KO.get(top, top)}"
            f"  {min(temps):.1f}°C ~ {max(temps):.1f}°C"
        )
    return "\n".join(lines)


# ──────────────────────────────────────────────
# Tools
# ──────────────────────────────────────────────

@tool
def get_current_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다. city는 한국어·영어 모두 가능합니다 (예: 서울, Tokyo)."""
    if not OPENWEATHER_API_KEY:
        return "⚠️ OPENWEATHER_API_KEY 미설정"
    geo = _geocode(city)
    if not geo:
        return f"❌ '{city}' 도시를 찾을 수 없습니다."
    lat, lon, name = geo
    try:
        r = requests.get(
            f"{OWM_BASE}/weather",
            params={"lat": lat, "lon": lon, "appid": OPENWEATHER_API_KEY,
                    "units": "metric", "lang": "kr"},
            # timeout=10,
        )
        r.raise_for_status()
        return _fmt_current(r.json(), name)
    except Exception as e:
        return f"❌ 오류: {e}"


@tool
def get_weather_forecast(city: str, days: int = 3) -> str:
    """도시의 날씨 예보를 조회합니다. days는 1~5일 (기본 3일)."""
    if not OPENWEATHER_API_KEY:
        return "⚠️ OPENWEATHER_API_KEY 미설정"
    days = max(1, min(days, 5))
    geo  = _geocode(city)
    if not geo:
        return f"❌ '{city}' 도시를 찾을 수 없습니다."
    lat, lon, name = geo
    try:
        r = requests.get(
            f"{OWM_BASE}/forecast",
            params={"lat": lat, "lon": lon, "appid": OPENWEATHER_API_KEY,
                    "units": "metric", "lang": "kr", "cnt": days * 8},
            timeout=10,
        )
        r.raise_for_status()
        return _fmt_forecast(r.json(), name, days)
    except Exception as e:
        return f"❌ 오류: {e}"


@tool
def compare_weather(city1: str, city2: str) -> str:
    """두 도시의 현재 날씨를 나란히 비교합니다."""
    r1 = get_current_weather.invoke({"city": city1})
    r2 = get_current_weather.invoke({"city": city2})
    return f"{r1}\n\n{r2}"


# ──────────────────────────────────────────────
# LLM
# ──────────────────────────────────────────────

def build_llm() -> ChatOpenAI:
    if ELICE_URL and ELICE_API_KEY:
        base_url, api_key = ELICE_URL, ELICE_API_KEY
    else:
        base_url, api_key = OPENROUTER_URL, OPENROUTER_API_KEY

    return ChatOpenAI(
        model           = MODEL_NAME,
        openai_api_key  = api_key,
        openai_api_base = base_url,
        temperature     = 0.3,
    )


# ──────────────────────────────────────────────
# 대화 루프
# ──────────────────────────────────────────────
SYSTEM_PROMPT = (
    "당신은 친절한 날씨 도우미입니다. "
    "사용자의 요청에 따라 현재 날씨 조회, 날씨 예보, 두 도시 비교를 수행하세요. "
    "날씨 데이터를 받으면 핵심 정보를 자연스럽게 요약하고, "
    "외출 여부·옷차림 등 실용적인 조언도 덧붙여 주세요."
)


def run_chatbot() -> None:
    print("=" * 55)
    print("  🌤️  날씨 챗봇 (LangGraph + OpenRouter)  🌤️")
    print("=" * 55)
    print("  예) '서울 날씨 어때?'")
    print("  예) '도쿄 3일 예보 알려줘'")
    print("  예) '서울과 부산 날씨 비교해줘'")
    print("  종료: q / quit")
    print("=" * 55)

    tools  = [get_current_weather, get_weather_forecast, compare_weather]
    agent  = create_agent(build_llm(), tools, system_prompt=SYSTEM_PROMPT)
    history: list = []

    while True:
        try:
            user_input = input("\n사용자: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n👋 챗봇을 종료합니다.")
            break

        if not user_input:
            continue
        if user_input.lower() in {"q", "quit", "exit", "종료"}:
            print("👋 챗봇을 종료합니다.")
            break

        history.append(HumanMessage(content=user_input))

        try:
            result   = agent.invoke({"messages": history})
            messages = result["messages"]
            answer   = messages[-1].content
            print(f"\n챗봇: {answer}")

            # 최근 20개 메시지만 유지
            history = messages[-20:]

        except Exception as e:
            print(f"\n⚠️ 오류: {e}")
            history = history[:-1]   # 오염 방지


if __name__ == "__main__":
    run_chatbot()

  🌤️  날씨 챗봇 (LangGraph + OpenRouter)  🌤️
  예) '서울 날씨 어때?'
  예) '도쿄 3일 예보 알려줘'
  예) '서울과 부산 날씨 비교해줘'
  종료: q / quit

챗봇: 안녕하세요! 날씨 도우미입니다. 😊

혹시 날씨 정보가 필요하신가요? 다음과 같이 도와드릴 수 있어요:

1. **현재 날씨 조회** - 특정 도시의 현재 날씨를 알려드려요
2. **날씨 예보** - 앞으로 1~5일간의 날씨 예보를 확인할 수 있어요
3. **두 도시 비교** - 두 도시의 날씨를 나란히 비교해드려요

어떤 도시의 날씨를 알고 싶으신가요? 예를 들어 "서울 날씨 알려줘" 또는 "부산과 대구 날씨 비교해줘"라고 말씀해 주세요!
👋 챗봇을 종료합니다.


In [39]:
from langchain.agents import create_agent
help(create_agent)

Help on function create_agent in module langchain.agents.factory:

create_agent(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]'
    Creates an agent graph that calls tools in a loop until a stopping condition is met.
    
    For more details on using `create_agent`,
    visit the [A

In [40]:
_geocode('서울')

서울 37.5666791 126.9782914


(37.5666791, 126.9782914, '서울')

In [41]:
city = '서울'


"""도시명 → (위도, 경도, 표시명). 실패 시 None."""
r = requests.get(
    f"{OWM_GEO}/direct",
    params={"q": city, "limit": 5, "appid": OPENWEATHER_API_KEY},
    timeout=10,
)
r.raise_for_status()
data = r.json()
# if not data:
#     return None
loc  = data[0]
name = loc.get("local_names", {}).get("ko") or loc["name"]
print(name, loc["lat"], loc["lon"])
# return loc["lat"], loc["lon"], name

서울 37.5666791 126.9782914


In [42]:
OWM_BASE = "https://api.openweathermap.org/data/2.5"

res = requests.get(
    f"{OWM_BASE}/weather",
    params={"lat": loc["lat"], "lon": loc["lon"], "appid": OPENWEATHER_API_KEY,
            "units": "metric", "lang": "kr"},
    # timeout=10,
)

res.raise_for_status()

In [43]:
res.json()

{'coord': {'lon': 126.979, 'lat': 37.5657},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': '온흐림',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 21.53,
  'feels_like': 21.67,
  'temp_min': 21.53,
  'temp_max': 21.53,
  'pressure': 1014,
  'humidity': 74,
  'sea_level': 1014,
  'grnd_level': 1005},
 'visibility': 10000,
 'wind': {'speed': 5.51, 'deg': 223, 'gust': 7.19},
 'clouds': {'all': 97},
 'dt': 1778473597,
 'sys': {'country': 'KR', 'sunrise': 1778444812, 'sunset': 1778495409},
 'timezone': 32400,
 'id': 1835848,
 'name': 'Seoul',
 'cod': 200}